# Evidence: Word to PDF

**What it does**
Converts evidence documents from Word (`.docx`) to PDF. It lists the files in the evidences folder, converts one file or all of them, and writes each PDF next to its `.docx` (`EV-01.docx` gives `EV-01.pdf`).

**Why**
Gemini and Claude read PDFs, not Word files. Evidence often arrives as `.docx`, so it has to become a PDF before the AI can judge test steps against it (the assessment). The conversion lives in the app's `DocxConverter` service (`services/evidences/docx_converter.py`), so the API can use the same code later.

**How to use**
1. Put `.docx` files in the evidences folder: `storage/evidences/` by default, or the folder set in `KCT_EVIDENCES_ROOT` (`.env`). Subfolders are fine. No files yet? Run the "Sample evidence" cell to create one.
2. Run all cells. To convert a single file, put its name in `FILE` (step 3) and run that cell.
3. Open the PDF from the folder to check it (in VS Code: right-click > Reveal in Finder; in JupyterLab: double-click it).

**Two conversion engines**
- `libreoffice`: keeps the Word layout exactly. Used automatically when LibreOffice is installed.
- `python`: needs only the pip packages `python-docx` and `reportlab`, so it works on a workbench without LibreOffice. Keeps text, headings, bold and italic, lists, tables and pictures, but not the exact page layout (columns, text boxes, headers and footers are not carried over).

`ENGINE = "auto"` picks LibreOffice when it is there, `python` otherwise.

**Needs**: `python-docx` and `reportlab` (`pip install python-docx reportlab`, or Step 3 of `01_setup.ipynb`).

## Setup

Finds the project folder, reads `.env`, and shows the evidences folder and which engine `auto` will use.

In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from services.app import boot
from services.evidences import libreoffice

kct = boot()
evidences = kct.evidences

ENGINE = "auto"  # auto | libreoffice | python

print("Evidences folder:", evidences.root)
print("LibreOffice:     ", libreoffice.binary() or "not installed, so 'auto' uses the python engine")

## Step 1: Sample evidence (optional)

Creates `EV-01_password_reset_tickets_SAMPLE.docx` in the evidences folder with `DocxFactory` (`factories/docx_factory.py`): made-up data for the sample case, with headings, a list, a table and a screenshot. Skip it when you have your own files. Running it again overwrites the sample.

In [ ]:
from factories import DocxFactory

print("Created", DocxFactory().create(evidences.root))

## Step 2: Files in the evidences folder

Every file, including subfolders, with its size. A `.docx` marked "needs PDF" has no PDF yet, or the `.docx` changed after its PDF was made.

In [ ]:
files = evidences.files()
if not files:
    print("No files in", evidences.root, "- add .docx files or run Step 1")
for relative in files:
    path = evidences.path(relative)
    note = ""
    if path.suffix.lower() == ".docx":
        pdf = path.with_suffix(".pdf")
        note = "PDF ok" if pdf.is_file() and pdf.stat().st_mtime >= path.stat().st_mtime else "needs PDF"
    print(f"{str(relative):<60} {path.stat().st_size / 1024:>8.1f} KB  {note}")

## Step 3: Convert one file

Set `FILE` to a name from the list above (relative to the evidences folder, for example `subfolder/EV-02.docx`).

In [ ]:
FILE = "EV-01_password_reset_tickets_SAMPLE.docx"

try:
    result = evidences.convert_to_pdf(FILE, engine=ENGINE)
except Exception as exc:
    print("❌", type(exc).__name__, exc)
else:
    print(f"✅ {result.pdf}  ({result.pdf.stat().st_size / 1024:.1f} KB, engine: {result.engine})")

## Step 4: Convert everything that needs it

Converts every `.docx` in the folder (and subfolders) whose PDF is missing or older than the `.docx`. Set `FORCE = True` to convert all of them again, for example after switching engine.

In [ ]:
FORCE = False

converted = evidences.convert_all_word(force=FORCE) if ENGINE == "auto" else [
    evidences.convert_to_pdf(p, engine=ENGINE) for p in evidences.files("*.docx")
    if FORCE or not evidences.path(p).with_suffix(".pdf").is_file()
]
if not converted:
    print("✅ Every .docx already has an up-to-date PDF")
for result in converted:
    print("✅", result.pdf.relative_to(evidences.root), f"({result.engine})")

## Step 5: Ready for the AI

The PDFs as the AI will receive them: each becomes a `Document` attachment, the same type the interpretation's agents get for the control details and the test plan. The assessment will send these with the test steps.

In [ ]:
from services.ai.attachments import Document

attachments = [Document.from_path(evidences.path(p)) for p in evidences.files("*.pdf")]
for doc in attachments:
    print(f"{doc.title:<60} {len(doc.data) / 1024:>8.1f} KB  {doc.mime_type}")